# App Clustering Pipeline: KMeans Approach

This notebook explores clustering ~4 000 mobile subscription apps into competitor groups using KMeans.

**Pipeline:**
1. **Data loading** — load and clean the raw app dataset
2. **Text preparation** — combine `trackName`, `overview`, and `features` into a single structured text per app
3. **Sentence embedding** — encode app features with `BAAI/bge-base-en`
4. **KMeans clustering** — partition the embedding space into `k` clusters
5. **UMAP visualization** — project to 2D for visual inspection

> **Note:** Unlike the graph-based approach in `NearestNeighbors.ipynb`, KMeans requires pre-specifying `k` and assumes convex, roughly equal-sized clusters. It is faster but less flexible for discovering natural groupings.

In [19]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import umap
import plotly.express as px

## 1. Data Loading & Cleaning

Load the raw JSON dataset and drop URL columns that carry no semantic signal for clustering.

In [20]:
df = pd.read_json('subscription_apps.json')
df.head()

,trackName,description,screenshotUrls,ipadScreenshotUrls,appletvScreenshotUrls,overview,features
0,Logo Maker - AI Art Design,Logo Maker AI-Powered Logo Design Made Easy\nC...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],[],Logo Maker is an AI-powered logo design app th...,[AI-Powered Logo Generation with customizable ...
1,Arvin® - AI Logo Maker,Arvin®: Your AI-Powered Logo Maker\n\nTransfor...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],[],Arvin® is an AI-powered logo maker designed fo...,"[AI-generated logo design, Customizable letter..."
2,AI Logo Generator & Design,AI LOGO MAKER: GENERATE STUNNING LOGOS IN SECO...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],AI Logo Generator is a powerful app designed f...,"[AI-powered logo generation in seconds, Over 5..."
3,Logo Maker _ AI Design Creator,Logo Maker allows you to create the logo of yo...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],Logo Maker is a user-friendly app designed for...,"[Access to over 7000 logo templates, Diverse c..."
4,Logo Maker Shop: AI Creator,Logo Maker Shop iOS app lets you create a stun...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],Logo Maker Shop is an intuitive iOS app design...,[10K+ customizable logo templates across 13 ca...


In [21]:
df.drop(columns=['screenshotUrls', 'ipadScreenshotUrls', 'appletvScreenshotUrls'], inplace=True)

## 2. Feature Embedding

Encode each app as a dense vector using `BAAI/bge-base-en` (768-dim).  
Embeddings are L2-normalised so cosine similarity equals the dot product — important for KMeans on directional data.

Embeddings are saved to disk after encoding so this expensive step (~5 min) can be skipped on re-runs.

In [ ]:
# Load the pre-trained sentence transformer (768-dim, optimised for semantic similarity)
model = SentenceTransformer("BAAI/bge-base-en")

# Use the features column — most structured signal for competitor detection
texts = df["features"].fillna("").tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2-normalise: cosine sim = dot product
)

df["embedding_features"] = embeddings.tolist()
df.head()

In [ ]:
#np.save("embeddings_features.npy", embeddings)

In [22]:
embeddings = np.load("embeddings_features.npy")
df["embedding_features"] = embeddings.tolist()
df.head()

,trackName,description,overview,features,embedding_features
0,Logo Maker - AI Art Design,Logo Maker AI-Powered Logo Design Made Easy\nC...,Logo Maker is an AI-powered logo design app th...,[AI-Powered Logo Generation with customizable ...,"[0.026286382228136063, 0.014646739698946476, -..."
1,Arvin® - AI Logo Maker,Arvin®: Your AI-Powered Logo Maker\n\nTransfor...,Arvin® is an AI-powered logo maker designed fo...,"[AI-generated logo design, Customizable letter...","[0.029506200924515724, 0.018569650128483772, 0..."
2,AI Logo Generator & Design,AI LOGO MAKER: GENERATE STUNNING LOGOS IN SECO...,AI Logo Generator is a powerful app designed f...,"[AI-powered logo generation in seconds, Over 5...","[0.02679363079369068, 0.02018563449382782, -0...."
3,Logo Maker _ AI Design Creator,Logo Maker allows you to create the logo of yo...,Logo Maker is a user-friendly app designed for...,"[Access to over 7000 logo templates, Diverse c...","[0.01727970503270626, 0.016116071492433548, 0...."
4,Logo Maker Shop: AI Creator,Logo Maker Shop iOS app lets you create a stun...,Logo Maker Shop is an intuitive iOS app design...,[10K+ customizable logo templates across 13 ca...,"[0.013803043402731419, 0.02349875494837761, 0...."


## 3. KMeans Clustering

Cluster apps into `k = 700` groups using KMeans on the L2-normalised embeddings.

**Why k = 700?**  
With ~4 264 apps and a target of ~5–6 apps per cluster (a reasonable competitor group size), `k ≈ 700` is a natural starting point.  
KMeans on normalised embeddings approximates spherical k-means (cosine similarity), which suits directional embedding spaces.

**Trade-offs vs. graph-based clustering (NearestNeighbors.ipynb):**
- ✅ Much faster to run
- ✅ Every app is guaranteed a cluster — no noise label
- ❌ Requires pre-specifying `k` 
- ❌ Assumes clusters are convex and similarly sized

In [23]:
from sklearn.cluster import KMeans

# Embeddings are already L2-normalised — KMeans on normalised vectors
# approximates spherical k-means (cosine similarity objective)
X_norm = embeddings

# k = 700 targets ~6 apps per cluster on average (4264 / 700 ≈ 6)
# Adjust k to trade off between cluster granularity and size
k = 700

kmeans = KMeans(n_clusters=k, random_state=42).fit(X_norm)
df['cluster'] = kmeans.labels_

print(f"Clusters: {k}  |  Apps: {len(df)}  |  Avg cluster size: {len(df)/k:.1f}")

Clusters: 700  |  Apps: 4264  |  Avg cluster size: 6.1


In [24]:
df['cluster'].value_counts()

cluster
267    29
257    28
160    27
35     26
493    23
       ..
115     1
540     1
522     1
438     1
666     1
Name: count, Length: 700, dtype: int64

## 4. Cluster Inspection

Spot-check individual apps and clusters to verify the clustering makes semantic sense before proceeding to visualisation.

In [25]:
df.iloc[192]

trackName                                       PAW Patrol Rescue World
description           Play as your favorite PAW Patrol™ pup and save...
overview              PAW Patrol™ Rescue World is an interactive adv...
features              [Play as popular PAW Patrol™ characters with u...
embedding_features    [-0.016508031636476517, -0.026263656094670296,...
cluster                                                             213
Name: 192, dtype: object

In [26]:
df[df['cluster'] == 666]

,trackName,description,overview,features,embedding_features,cluster
4259,Currency Converter Calculator•,"Introducing Currency Converter Calculator, you...",Currency Converter Calculator is a user-friend...,"[Offline Mode for access without internet, Liv...","[-0.003008872037753463, 0.00581491831690073, 0...",666


## 5. UMAP Visualization

Project the 768-dim embeddings to 2D using UMAP to visually inspect cluster structure.  
Points close together in the 2D plot were neighbours in the original embedding space.  
The interactive Plotly scatter allows hovering over points to see the app name.

In [27]:
umap_model = umap.UMAP(
    n_neighbors=15,    # controls local vs. global structure balance
    min_dist=0.1,      # minimum distance between points in 2D; lower = tighter clusters
    n_components=2,
    random_state=42,
    metric='cosine',   # consistent with how embeddings were compared during clustering
)
X_umap = umap_model.fit_transform(X_norm)

df['umap_x'] = X_umap[:, 0]
df['umap_y'] = X_umap[:, 1]

c:\Study\selfeducation\Universe_Group_test_task\.venv\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Study\selfeducation\Universe_Group_test_task\.venv\lib\site-packages\numba\np\ufunc\parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12040. The TBB threading layer is disabled.
  warnings.warn(problem)


In [ ]:
fig = px.scatter(
    df,
    x='umap_x',
    y='umap_y',
    color='cluster',               # колір за кластером
    hover_data=['trackName'],      # показуємо назву апки при наведенні
    title="UMAP visualization of app embeddings",
    width=1000,
    height=800,
    color_continuous_scale=px.colors.qualitative.Alphabet
)

fig.show(renderer="browser")

NameError: name 'px' is not defined